# 911 Calls Capstone Project

This project analyzes historical 911 emergency call data to uncover temporal, spatial, and operational patterns in emergency incidents. The analysis includes exploratory data analysis, emergency type prediction using machine learning, hotspot detection through clustering, anomaly detection for identifying unusual emergency demand, and geospatial visualization of incident locations. The goal is to derive actionable insights that can assist emergency services in resource allocation and strategic planning.

The datase is from [Kaggle](https://www.kaggle.com/mchirico/montcoalert).

## Techniques Used
- Data Cleaning and Feature Engineering
- Exploratory Data Analysis (EDA)
- Random Forest Classification
- K-Means Clustering
- Z-Score Based Anomaly Detection
- Geospatial Visualization using Plotly

## Technologies Used
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn
- Plotly

## Outcomes
- Identified high-risk ZIP codes and emergency hotspots.
- Detected abnormal spikes in emergency call demand.
- Predicted emergency categories using machine learning techniques.
- Visualized spatial distributions of incidents for resource planning.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
import plotly.express as px

Reading the file and cleaning the dataset

In [ ]:
df=pd.read_csv("911.csv")

In [ ]:
df.dropna()

Finding the reasons of the 911 calls from the title section

In [ ]:
df['Reason']=df['title'].apply(lambda x:x.split(":")[0])

In [ ]:
df

Finding the number of each reasons

In [ ]:
df['Reason'].value_counts()

Countplot to show the number of reasons

In [ ]:
sns.countplot(x='Reason',data=df)

Finding the datatype of the timeStamp column

In [ ]:
type(df['timeStamp'].loc[0])

Converted the timeStamp column to datetime

In [ ]:
df['timeStamp']=pd.to_datetime(df['timeStamp'])

Adding 3 more columns that are hour, month and day of the week to the dataframe

In [ ]:
time = df['timeStamp'].iloc[0]

In [ ]:
df['Hour']=df['timeStamp'].apply(lambda time:time.hour)
df['Month']=df['timeStamp'].apply(lambda time:time.month)
df['DayOfWeek']=df['timeStamp'].apply(lambda time:time.dayofweek)

In [ ]:
df

Changing the day of the week to names of the day of the week

In [ ]:
dmap = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}

In [ ]:
df['DayOfWeek']=df['DayOfWeek'].map(dmap)

In [ ]:
df

Count Plot for the the number of 911 calls that were made in a week on the basis of their reasons

In [ ]:
sns.countplot(x='Month',data=df,hue='Reason')

Some months are missing in the countplot above so we will create another dataframe 

In [ ]:
byMonth = df.groupby('Month').count()
byMonth

In [ ]:
byMonth['twp'].plot()

Linear fit for the number of calls per month

In [ ]:
sns.lmplot(x='Month',y='twp',data=byMonth.reset_index())

Created a date column 

In [ ]:
df['Date']=df['timeStamp'].apply(lambda time:time.date())
df.head()

Date column with the count() aggregate and create a plot of counts of 911 calls.

In [ ]:
df.groupby('Date').count()['twp'].plot();
plt.xticks(rotation=90);

Finding unique reasons of the 911 call and then plotting a line graph for each reason

In [ ]:
df['Reason'].unique()

In [ ]:
df[df['Reason']=='Traffic'].groupby('Date').count()['twp'].plot()
df[df['Reason']=='Fire'].groupby('Date').count()['twp'].plot()
df[df['Reason']=='EMS'].groupby('Date').count()['twp'].plot()
plt.xticks(rotation=90);

Creating heatmaps with seaborn and our data. We'll first need to restructure the dataframe so that the columns become the Hours and the Index becomes the Day of the Week. T

In [ ]:
dayHour=df.groupby(by=['DayOfWeek','Hour']).count()['Reason'].unstack()
dayHour.head()

Creating a heatmap

In [ ]:
sns.heatmap(dayHour,cmap='viridis')

**Emergency Type Prediction**

In [ ]:
X = df[['Hour','Month','zip']]
y = df['Reason']
le = LabelEncoder()
y = le.fit_transform(y)
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = RandomForestClassifier()
model.fit(X_train,y_train)

pred = model.predict(X_test)

In [ ]:
print(classification_report(y_test,pred))

**Emergency Hotspot Detection**

In [ ]:
zip_calls = df.groupby('zip').size().reset_index(name='total_calls')
kmeans = KMeans(n_clusters=4, random_state=42)
zip_calls['cluster'] = kmeans.fit_predict(zip_calls[['total_calls']])
zip_calls

In [ ]:
zip_calls.groupby('cluster')['total_calls'].sum()

**Anomaly Detection**

In [ ]:
daily_calls = df.resample('D', on='timeStamp').size()
from scipy.stats import zscore
daily_z = zscore(daily_calls)
anomalies = daily_calls[abs(daily_z) > 3]
print(anomalies)

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(daily_calls.index,daily_calls.values)
plt.scatter(anomalies.index,anomalies.values,color='red',s=100)
plt.title('Anomaly Detection in Daily 911 Calls')
plt.xlabel('Date')
plt.ylabel('Number of Calls')
plt.show()

**Interactive Emergency Location Map**

In [ ]:
fig = px.scatter_mapbox(df,lat='lat',lon='lng',color='Reason',hover_data=['zip', 'twp', 'title'],zoom=8,height=700);
fig.update_layout(mapbox_style='open-street-map',title='911 Emergency Calls by Location');
fig.show();